# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets in the dataset with their @id and field details.
print("Available record sets and fields:")
record_sets = []
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set['@id']}")
    record_sets.append(record_set['@id'])
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            # Each field is a dict or reference
            if isinstance(field, dict) and '@id' in field:
                print(f"    - Field @id: {field['@id']}")
            elif isinstance(field, str):
                print(f"    - Field @id: {field}")
    else:
        print("  (No fields listed)")
    print()

# For demonstration, display available record sets parsed.
print("Parsed record set @ids:")
print(record_sets)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use entity `@id`s identified in the overview above.

In [ ]:
# Select the available record set(s) by @id
# Here, we use all discovered record sets; you can also pick one if preferred
record_sets_ids = record_sets  # Use the discovered record set @ids from above

dataframes = {}
for record_set_id in record_sets_ids:
    # Load all records from the record set using its @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set: {record_set_id}")
        print("Columns:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for record set: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, and grouping data. All columns referenced by their `@id`s.

In [ ]:
# If there is at least one non-empty dataframe, proceed
if dataframes:
    # For demonstration, select first available record set
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]

    # Find a numeric field (try to infer automatically)
    numeric_field_id = None

    for col in df.columns:
        # Try to infer numeric column by dtype
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    
    if numeric_field_id is not None:
        print(f"Numeric field for demonstration: {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.75)  # For demonstration, top 25%
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by a non-numeric field
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric field found for grouping.")
    else:
        print("No numeric field detected in the first record set for EDA demonstration.")
else:
    print("No dataframes loaded. Please check that the dataset contains record sets with records.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization example: Histogram of the numeric field (if found in previous step)
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[record_set_id]
    if numeric_field_id is not None and numeric_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No visualization possible because no data is loaded.")

## 6. Conclusion
This notebook demonstrated how to programmatically explore and analyze the FAIR² dataset using the `mlcroissant` library and referenced all dataset entities—including record sets, fields, and columns—by their `@id` keys. You can adapt these steps to process additional Croissant datasets or perform deeper statistical and domain-specific analyses.